In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

In [3]:
titanic = sns.load_dataset('titanic')
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [4]:
titanic.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [5]:
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X = titanic[features]
y = titanic['survived'].values

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [7]:
age_median = X_train['age'].median()
embarked_mode = X_train['embarked'].mode()[0] # la valeur qui apparait le + souvent

X_train = X_train.fillna({'age': age_median, 'embarked': embarked_mode})
X_test = X_test.fillna({'age': age_median, 'embarked': embarked_mode})

print("Valeurs manquantes restantes :", X_train.isna().sum().sum(), X_test.isna().sum().sum())

Valeurs manquantes restantes : 0 0


In [8]:
cat = ['sex', 'embarked', 'pclass']
X_train = pd.get_dummies(X_train, columns=cat, drop_first=True)

In [10]:
X_test = pd.get_dummies(X_test, columns=cat, drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

X_train.head()

,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S,pclass_2,pclass_3
692,28.5,0,0,56.4958,True,False,True,False,True
481,28.5,0,0,0.0000,True,False,True,True,False
527,28.5,0,0,221.7792,True,False,True,False,False
855,18.0,0,1,9.3500,False,False,True,False,True
801,31.0,1,1,26.2500,False,False,True,True,False


In [11]:
num = ['age', 'fare']
scaler = StandardScaler()
X_train[num] = scaler.fit_transform(X_train[num])
X_test[num] = scaler.transform(X_test[num])

X_train.head()

,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S,pclass_2,pclass_3
692,-0.081135,0,0,0.513812,True,False,True,False,True
481,-0.081135,0,0,-0.662563,True,False,True,True,False
527,-0.081135,0,0,3.955399,True,False,True,False,False
855,-0.887827,0,1,-0.467874,False,False,True,False,True
801,0.110934,1,1,-0.115977,False,False,True,True,False


In [12]:
feature_names = list(X_train.columns)
X_train = X_train.values.astype('float32')
X_test = X_test.values.astype('float32')

In [13]:
print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)
print("Shape X_test:", X_test.shape)

Shape X_train: (712, 9)
Shape y_train: (712,)
Shape X_test: (179, 9)


In [14]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(27, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 27)             │           270 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            28 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 298 (1.16 KB)

 Trainable params: 298 (1.16 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [16]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32,
                    validation_split=0.1, callbacks=[early_stop])

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5328 - loss: 0.7193 - val_accuracy: 0.6250 - val_loss: 0.6440
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6984 - loss: 0.6637 - val_accuracy: 0.7917 - val_loss: 0.6049
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7344 - loss: 0.6253 - val_accuracy: 0.8194 - val_loss: 0.5756
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7234 - loss: 0.5969 - val_accuracy: 0.7639 - val_loss: 0.5524
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7250 - loss: 0.5751 - val_accuracy: 0.7778 - val_loss: 0.5328
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7297 - loss: 0.5581 - val_accuracy: 0.7917 - val_loss: 0.5167
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7437 - loss: 0.5442 - val_accuracy: 0.7778 - val_loss: 0.5034
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7547 - loss: 0.5322 - val_accuracy: 0.7778 - val_loss

In [17]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy : {accuracy*100:.2f}%")

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8101 - loss: 0.4561 
Test Accuracy : 81.01%


In [18]:
predictions = model.predict(X_test)
y_pred = (predictions > 0.5).astype(int).ravel()

df_test = pd.DataFrame(X_test, columns=feature_names)
df_test[num] = scaler.inverse_transform(df_test[num])

df_test['age'] = df_test['age'].round(0).astype(int)
df_test['sibsp'] = df_test['sibsp'].astype(int)
df_test['parch'] = df_test['parch'].astype(int)
df_test['fare'] = df_test['fare'].round(2)

df_test['True_Class'] = y_test
df_test['Predicted_Class'] = y_pred
df_test['Proba'] = predictions.ravel().round(3)
df_test['correct'] = df_test['True_Class'] == df_test['Predicted_Class']

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 


In [19]:
df_test.head(10)

,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S,pclass_2,pclass_3,True_Class,Predicted_Class,Proba,correct
0,24,2,0,24.150000,1.0,0.0,1.0,0.0,1.0,0,0,0.138,True
1,44,0,1,16.100000,1.0,0.0,1.0,0.0,1.0,0,0,0.072,True
2,22,0,0,7.230000,1.0,0.0,0.0,0.0,1.0,1,0,0.215,False
3,41,2,0,14.110000,1.0,0.0,1.0,0.0,1.0,0,0,0.130,True
4,28,1,0,15.500000,0.0,1.0,0.0,0.0,1.0,1,1,0.685,True
5,36,1,0,17.400000,0.0,0.0,1.0,0.0,1.0,1,0,0.483,False
6,36,0,0,13.000000,0.0,0.0,1.0,1.0,0.0,1,1,0.813,True
7,45,0,0,26.549999,1.0,0.0,1.0,0.0,0.0,0,0,0.215,True
8,49,1,1,110.879997,1.0,0.0,0.0,0.0,0.0,0,0,0.414,True
9,34,1,0,26.000000,1.0,0.0,1.0,1.0,0.0,0,0,0.183,True


In [20]:
df_test['correct'].value_counts()

correct
True     145
False     34
Name: count, dtype: int64

In [21]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.79      0.94      0.86       110
           1       0.86      0.61      0.71        69

    accuracy                           0.81       179
   macro avg       0.82      0.77      0.79       179
weighted avg       0.82      0.81      0.80       179



Le modèle est très bon pour retrouver la classe 0 (recall = 94 %).

Pour la classe 1, lorsqu'il prédit 1, il a généralement raison (precision = 86 %), mais il n'ose pas suffisamment prédire 1 : il ne détecte que 61 % des vrais 1. Il rate une proportion assez importante des survivants